In [1]:
!pip install boto3 model-registry --quiet

In [2]:
import os
from dotenv import load_dotenv

local_base_dir = os.path.expanduser("~/shared")
job_name = os.environ.get("JOB_NAME", "sft-llama-3-2-1b")
params_file = f"{local_base_dir}/{job_name}-output-1.env"

load_dotenv(params_file)

True

In [3]:
import os
import boto3
import botocore

aws_access_key_id = os.environ.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
endpoint_url = os.environ.get('AWS_S3_ENDPOINT')
region_name = os.environ.get('AWS_DEFAULT_REGION')
bucket_name = os.environ.get('AWS_S3_BUCKET')

session = boto3.session.Session(aws_access_key_id=aws_access_key_id,
                                aws_secret_access_key=aws_secret_access_key)

s3_resource = session.resource(
    's3',
    config=botocore.client.Config(signature_version='s3v4'),
    endpoint_url=endpoint_url,
    region_name=region_name)

bucket = s3_resource.Bucket(bucket_name)

#upload the model directory without git
def upload_directory_to_s3(local_directory, s3_prefix):
    for root, dirs, files in os.walk(local_directory):
        for filename in files:
            file_path = os.path.join(root, filename)
            relative_path = os.path.relpath(file_path, local_directory)
            if ".git" in relative_path:
                continue
            s3_key = os.path.join(s3_prefix, relative_path)
            print(f"{file_path} -> {s3_key}")
            bucket.upload_file(file_path, s3_key)


def list_objects(prefix):
    filter = bucket.objects.filter(Prefix=prefix)
    for obj in filter.all():
        print(obj.key)

In [10]:
import re
from model_registry._client import ModelRegistry, StoreError

def get_tuned_model_name(model_path: str) -> str:
    match = re.match(r"(.+?)-(\d+\.\d+-.*)", model_path)
    if match:
        return match.group(1)
    else:
        return model_path

def get_version_prefix(model_path: str) -> str:
    match = re.match(r"(.+?)-(\d+\.\d+-.*)", model_path)
    if match:
        return match.group(2) + '-'
    else:
        return ""

def get_next_tuned_version_name(
    registry: ModelRegistry,
    model_name: str,
    prefix: str,
):
    try:
        versions = list(registry.get_model_versions(model_name))
    except StoreError as e:
        if "does not exist" in str(e):
            return f"{prefix}1", 1

    def is_numeric_suffix(vname: str):
        m = re.search(rf"{re.escape(prefix)}(\d+)$", vname)
        if not m:
            return False
        suffix = m.group(1)
        return int(suffix) <= 99999999

    tuned_versions = [
        v for v in versions
        if getattr(v, "name", "").startswith(prefix) and is_numeric_suffix(getattr(v, "name", ""))
    ]

    if not tuned_versions:
        return f"{prefix}1", 1

    def extract_suffix(vname: str):
        m = re.search(rf"{re.escape(prefix)}(\d+)$", vname)
        return int(m.group(1)) if m else -1

    max_suffix = max(extract_suffix(getattr(v, "name", "")) for v in tuned_versions)
    next_version_num = max_suffix + 1
    next_version_name = f"{prefix}{next_version_num}"
    return next_version_name, next_version_num

In [11]:
model_name = os.environ.get("MODEL_NAME")
local_path = os.environ.get("TUNED_MODEL_LOCATION")
department = os.environ.get("DEPARTMENT", "bank")

s3_path_prefix = f"{model_name}-{department}"

In [12]:
model_registry_url = os.environ.get('MODEL_REGISTRY_URL')
model_registry_user_token = os.environ.get('OPENSHIFT_API_TOKEN')

model_registry_port = 443
model_author = department
tuned_model_name = get_tuned_model_name(s3_path_prefix) #"meta-llama/Llama"
version_prefix = get_version_prefix(s3_path_prefix) #"3.2-1B-Instruct-tuned-"

In [13]:
from model_registry import ModelRegistry, utils
import os
registry = ModelRegistry(
    server_address=model_registry_url,
    port=model_registry_port,
    author=model_author,
    user_token=model_registry_user_token
)

In [14]:
version, index = get_next_tuned_version_name(
    registry,
    model_name=tuned_model_name,
    prefix=version_prefix
)

In [15]:
s3_path = f"{s3_path_prefix}-{index}"
upload_directory_to_s3(str(local_path), s3_path)

/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model/tokenizer_config.json -> meta-llama/Llama-3.2-1B-Instruct-bank-2/tokenizer_config.json
/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model/model.safetensors -> meta-llama/Llama-3.2-1B-Instruct-bank-2/model.safetensors
/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model/config.json -> meta-llama/Llama-3.2-1B-Instruct-bank-2/config.json
/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model/generation_config.json -> meta-llama/Llama-3.2-1B-Instruct-bank-2/generation_config.json
/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model/special_tokens_map.json -> meta-llama/Llama-3.2-1B-Instruct-bank-2/special_tokens_map.json
/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model/chat_template.jinja -> meta-llama/Llama-3.2-1B-Instruct-bank-2/chat_template.jinja
/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model/tokeniz

In [16]:
list_objects(s3_path)

meta-llama/Llama-3.2-1B-Instruct-bank-2/chat_template.jinja
meta-llama/Llama-3.2-1B-Instruct-bank-2/config.json
meta-llama/Llama-3.2-1B-Instruct-bank-2/generation_config.json
meta-llama/Llama-3.2-1B-Instruct-bank-2/model.safetensors
meta-llama/Llama-3.2-1B-Instruct-bank-2/special_tokens_map.json
meta-llama/Llama-3.2-1B-Instruct-bank-2/tokenizer.json
meta-llama/Llama-3.2-1B-Instruct-bank-2/tokenizer_config.json


In [17]:
from dotenv import set_key, dotenv_values
from pathlib import Path

params_file = f"{local_base_dir}/{job_name}-output-2.env"
Path(params_file).write_text("")

set_key(params_file, "MODEL_PATH", s3_path)
set_key(params_file, "MODEL_NAME", tuned_model_name)
set_key(params_file, "MODEL_VERSION", version)

!cat {params_file}

MODEL_PATH='meta-llama/Llama-3.2-1B-Instruct-bank-2'
MODEL_NAME='meta-llama/Llama'
MODEL_VERSION='3.2-1B-Instruct-bank-2'
